In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes evaluate rouge_score

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 623.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 71.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 81.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 69.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 59.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 81.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66

In [ ]:
import torch
print(torch.__version__)

2.5.1+cu121


In [1]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.0 MB/s eta 0:00:00


In [14]:

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from huggingface_hub import login
import evaluate

In [15]:
class DatasetHandler:
    def __init__(self, dataset_name, sample_size=1000):
        self.dataset_name = dataset_name
        self.sample_size = sample_size

    def load_and_prepare(self):
        dataset = load_dataset(self.dataset_name, split="train")
        dataset = dataset.shuffle(seed=42).select(range(self.sample_size))


        def format_instruction(example):
            return {
                "text": f"User: {example['instruction']}\nAssistant: {example['response']}",
                "instruction": example["instruction"],
                "response": example["response"]
            }

        dataset = dataset.map(format_instruction)

        split_dataset = dataset.train_test_split(test_size=0.1, seed=42)

        return split_dataset["train"], split_dataset["test"]

In [16]:
class ModelHandler:
    def __init__(self, model_id):
        self.model_id = model_id

    def setup_tokenizer(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.tokenizer.padding_side = "right"
        return self.tokenizer

    def setup_model(self):
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            quantization_config=bnb_config,
            device_map="auto"
        )
        self.model.config.use_cache = False

        self.model = prepare_model_for_kbit_training(self.model)
        return self.model

    def apply_lora(self):
        peft_config = LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            lora_dropout=0.05,
            task_type="CAUSAL_LM"
        )

        self.model = get_peft_model(self.model, peft_config)
        return self.model



In [19]:
class TrainerHandler:
    def __init__(self, model, train_data):
        self.model = model
        self.train_data = train_data

    def setup_training_args(self):
        self.training_args = TrainingArguments(
            output_dir="./results",
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            learning_rate=2e-4,
            max_steps=200,
            bf16=True,
            logging_steps=10
        )

    def train(self):
        self.trainer = SFTTrainer(
            model=self.model,
            train_dataset=self.train_data,
            args=self.training_args,
        )

        print(" Training...")
        self.trainer.train()

    def save_model(self):
        self.trainer.model.save_pretrained("fine_tuned_model")

In [20]:
class ModelEvaluator:
    def __init__(self, base_model, ft_model, tokenizer, test_data):
        self.base_model = base_model
        self.ft_model = ft_model
        self.tokenizer = tokenizer
        self.test_data = test_data
        self.rouge = evaluate.load("rouge")

    def generate(self, model, prompt):
        inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=50)
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def evaluate(self, num_samples=10):
        preds_base, preds_ft, refs = [], [], []

        for i in range(num_samples):
            prompt = self.test_data[i]["instruction"]
            ref = self.test_data[i]["response"]

            text = f"User: {prompt}\nAssistant:"

            base = self.generate(self.base_model, text)
            ft = self.generate(self.ft_model, text)

            base = base.split("Assistant:")[-1].strip()
            ft = ft.split("Assistant:")[-1].strip()

            preds_base.append(base)
            preds_ft.append(ft)
            refs.append(ref)

        return self.compute_metrics(preds_base, preds_ft, refs)

    def compute_metrics(self, pb, pf, refs):
        return {
            "Base": {
                "ROUGE": self.rouge.compute(predictions=pb, references=refs),
                "BLEU": sum(sentence_bleu([r.split()], p.split()) for p, r in zip(pb, refs)) / len(refs)
            },
            "Fine-Tuned": {
                "ROUGE": self.rouge.compute(predictions=pf, references=refs),
                "BLEU": sum(sentence_bleu([r.split()], p.split()) for p, r in zip(pf, refs)) / len(refs)
            }
        }




In [32]:
class HumanEvaluator:
    def __init__(self, base_model, ft_model, tokenizer, test_data):
        self.base_model = base_model
        self.ft_model = ft_model
        self.tokenizer = tokenizer
        self.test_data = test_data

    def evaluate(self, num_samples=10):
        print(f"Performing human evaluation for {num_samples} samples...")
        print("This is a placeholder. In a real scenario, this would involve human review and annotation of model outputs.")
        # In a real scenario, you would implement logic here to collect human judgments
        # or present outputs for human review.
        return {"human_evaluation_status": "placeholder_evaluated", "num_samples": num_samples}

In [12]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [21]:
from nltk.translate.bleu_score import sentence_bleu

In [30]:

from peft import PeftModel

class Pipeline:
    def __init__(self):
        self.model_id = "Qwen/Qwen2.5-1.5B-Instruct"

    def run(self):

        # -----------------------------
        # Dataset
        # -----------------------------
        train_data, test_data = DatasetHandler(
            "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
        ).load_and_prepare()

        # -----------------------------
        # Model Setup
        # -----------------------------
        mh = ModelHandler(self.model_id)
        tokenizer = mh.setup_tokenizer()
        model = mh.setup_model()
        model = mh.apply_lora()

        # -----------------------------
        # Training
        # -----------------------------
        trainer = TrainerHandler(model, train_data)
        trainer.setup_training_args()
        trainer.train()

        # -----------------------------
        # Save model
        # -----------------------------
        trainer.save_model()
        tokenizer.save_pretrained("fine_tuned_model")

        # -----------------------------
        # Load base model
        # -----------------------------
        base_model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            device_map="auto"
        )

        # -----------------------------
        # Load fine-tuned model (LoRA)
        # -----------------------------
        ft_model = PeftModel.from_pretrained(
            base_model,
            "fine_tuned_model"
        )

        # -----------------------------
        # Evaluation
        # -----------------------------
        evaluator = ModelEvaluator(base_model, ft_model, tokenizer, test_data)

        results = evaluator.evaluate()

        print("\n FINAL RESULTS:")
        print(results)


# ==============================
# RUN
# ==============================
if __name__ == "__main__":
    pipeline = Pipeline()
    pipeline.run()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


 Training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.773361
20,1.479331
30,1.396339
40,1.259618
50,1.258425
60,1.184428
70,1.175250
80,1.165907
90,1.086919
100,1.136514


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)



 FINAL RESULTS:
{'Base': {'ROUGE': {'rouge1': np.float64(0.33239612488998593), 'rouge2': np.float64(0.12913193465932915), 'rougeL': np.float64(0.23549788648955255), 'rougeLsum': np.float64(0.2583692501265883)}, 'BLEU': 0.043591408403061374}, 'Fine-Tuned': {'ROUGE': {'rouge1': np.float64(0.30668229372221645), 'rouge2': np.float64(0.08654384224802622), 'rougeL': np.float64(0.18326610069022575), 'rougeLsum': np.float64(0.20928637550877188)}, 'BLEU': 0.00689643324162183}}


In [43]:
class HumanEvaluator:
    def __init__(self, base_model, ft_model, tokenizer, test_data):
        self.base_model = base_model
        self.ft_model = ft_model
        self.tokenizer = tokenizer
        self.test_data = test_data

    def generate(self, model, prompt):
        inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=50)
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def evaluate(self, num_samples=5):
        print("\n HUMAN EVALUATION (QUALITATIVE COMPARISON)\n")

        for i in range(num_samples):
            prompt = self.test_data[i]["instruction"]

            input_text = f"User: {prompt}\nAssistant:"

            base_out = self.generate(self.base_model, input_text)
            ft_out = self.generate(self.ft_model, input_text)

            base_out = base_out.split("Assistant:")[-1].strip()
            ft_out = ft_out.split("Assistant:")[-1].strip()

            print("\n==============================")
            print(f" Prompt: {prompt}")

            print("\n Base Model Response:")
            print(base_out)

            print("\n Fine-Tuned Model Response:")
            print(ft_out)

            # Simple qualitative observation
            if len(ft_out) > len(base_out):
                print("\n Observation: Fine-tuned model gives more detailed response.")
            else:
                print("\n Observation: Responses are similar or require manual review.")

        print("\n Human evaluation completed!")

if __name__ == "__main__":
    pipeline = HumanEvalPipeline()
    pipeline.run()

 Loading Data...
 Loading Tokenizer...
 Loading Models...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

 Starting Human Evaluation...


 HUMAN EVALUATION (QUALITATIVE COMPARISON)


 Prompt: I wwant assistance retrieving my profile password

 Base Model Response:
I'm here to help you retrieve your profile password! Let's get started:

1. First, please check if you've received an email with a link or instructions on how to reset your password. If so, follow the steps outlined in that email

 Fine-Tuned Model Response:
I'm here to help you retrieve your profile password! Please follow these steps:

1. Go to our website and navigate to the "Account" or "Profile" section.
2. Look for an option labeled "Forgot Password?" or something similar.

 Observation: Responses are similar or require manual review.

 Prompt: I want assistance seeing the bloody cancellation fee

 Base Model Response:
I'm sorry to hear that you're seeking assistance in viewing our cancellation fee. To get the information you need, could you please provide me with your account details or any other relevant information? Once

##Task-Specific Evaluator

In [39]:
class TaskEvalPipeline:
    def __init__(self):
        self.model_id = "Qwen/Qwen2.5-1.5B-Instruct"

    # -----------------------------
    # Load Dataset
    # -----------------------------
    def load_data(self):
        dataset_handler = DatasetHandler(
            "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
        )
        _, self.test_data = dataset_handler.load_and_prepare()

    # -----------------------------
    # Load Tokenizer
    # -----------------------------
    def load_tokenizer(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.tokenizer.padding_side = "right"

    # -----------------------------
    # Load Models
    # -----------------------------
    def load_models(self):
        from peft import PeftModel

        # Base model
        self.base_model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            dtype=torch.bfloat16 # Specify dtype if not already handled
        )
        if torch.cuda.is_available():
            self.base_model.to('cuda')

        # Fine-tuned model (LoRA)
        self.ft_model = PeftModel.from_pretrained(
            self.base_model,
            "fine_tuned_model"
        )

    # -----------------------------
    # Run Task-Specific Evaluation
    # -----------------------------
    def run_evaluation(self):
        evaluator = TaskSpecificEvaluator(
            self.base_model,
            self.ft_model,
            self.tokenizer,
            self.test_data
        )

        results = evaluator.evaluate(num_samples=20)

        print("\n TASK-SPECIFIC EVALUATION RESULTS\n")

        for model, metrics in results.items():
            print(f"🔹 {model}")
            print(f"   Intent Accuracy : {metrics['Intent Accuracy']:.4f}")
            print(f"   Keyword Match   : {metrics['Keyword Match']:.4f}")
            print("-" * 40)

        return results

    # -----------------------------
    # Main Run
    # -----------------------------
    def run(self):
        print(" Loading Data...")
        self.load_data()

        print(" Loading Tokenizer...")
        self.load_tokenizer()

        print(" Loading Models...")
        self.load_models()

        print(" Running Task-Specific Evaluation...\n")
        self.run_evaluation()

In [40]:
if __name__ == "__main__":
    pipeline = TaskEvalPipeline()
    pipeline.run()

 Loading Data...
 Loading Tokenizer...
 Loading Models...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

 Running Task-Specific Evaluation...


 TASK-SPECIFIC EVALUATION RESULTS

🔹 Base Model
   Intent Accuracy : 0.0000
   Keyword Match   : 0.2571
----------------------------------------
🔹 Fine-Tuned Model
   Intent Accuracy : 0.0000
   Keyword Match   : 0.2543
----------------------------------------
